# Dana's Bakery Order Pipeline with LangGraph

A deterministic Google Colab order system for Dana's Bakery in Nachlaot. It uses local Python rules only, with no LLM calls or API keys. Conditional graph routes send invalid or unfillable orders directly to rejection, while valid orders continue through delivery, pricing, inventory commitment, and a warm customer reply.

If a replacement is needed, the customer must approve it. Pass replies with the optional replacement_answers list. The six customer-facing output keys are status, item, delivery, total, message, and steps. For rejected orders, item is None because nothing was sold.


## 1. Install LangGraph if needed

The LangGraph package runs the workflow. Order validation, substitutions, inventory, and pricing are ordinary Python code.


In [ ]:
# Install LangGraph only if it is not already available in this Colab runtime.
try:
    import langgraph  # noqa: F401
    print('LangGraph is already installed.')
except ImportError:
    !pip -q install langgraph


## 2. Define the state, menu, and starting stock

Inventory is shared by accepted orders during the current runtime. Restarting the runtime restores the starting quantities.


In [ ]:
from typing import Literal, TypedDict

from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph


class OrderState(TypedDict, total=False):
    # The six required result keys:
    status: Literal['accepted', 'rejected']
    item: str | None
    delivery: Literal['pickup', 'delivery']
    total: float
    message: str
    steps: list[str]

    # Customer inputs and optional replacement answers:
    quantity: int | str
    day: str
    city: str
    replacement_answers: list[str | bool]

    # Internal state used by nodes:
    valid: bool
    requested_item: str
    selected_item: str
    decision: str
    reason: str
    offer_item: str
    offer_number: int
    substitution: bool
    delivery_fee: int
    outside_delivery_area: bool
    goods_subtotal: float
    discount: float


MENU = {
    'challah': 18,
    'babka': 45,
    'box of rugelach': 30,
    'sourdough loaf': 25,
}
STARTING_INVENTORY = {
    'challah': 10,
    'babka': 0,
    'box of rugelach': 3,
    'sourdough loaf': 4,
}
INVENTORY = STARTING_INVENTORY.copy()

# Follow this order when offering replacements, and make at most two offers.
REPLACEMENT = {
    'babka': 'box of rugelach',
    'box of rugelach': 'sourdough loaf',
    'sourdough loaf': 'challah',
}
WEEKDAYS = {'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'}
ALIASES = {
    'rugelach': 'box of rugelach',
    'box of rugelach': 'box of rugelach',
    'sourdough': 'sourdough loaf',
    'sourdough loaf': 'sourdough loaf',
}


## 3. Define the graph nodes

Each node appends its name to steps. Rejected paths skip delivery and pricing entirely. Only accepted orders reach the pricing and inventory-commit nodes.


In [ ]:
def add_step(state: OrderState, step_name: str) -> list[str]:
    """Return the existing diary steps with this node added at the end."""
    return [*state.get('steps', []), step_name]


def quantity_label(item: str, quantity: int) -> str:
    """Format a product and its quantity naturally for customer messages."""
    if item == 'box of rugelach':
        return f"{quantity} {'box' if quantity == 1 else 'boxes'} of rugelach"
    if item == 'sourdough loaf':
        return f"{quantity} sourdough {'loaf' if quantity == 1 else 'loaves'}"
    return f"{quantity} {item if quantity == 1 else item + 's'}"


def validate_order(state: OrderState) -> dict:
    steps = add_step(state, 'validate_order')
    raw_item = str(state.get('item', '')).strip().casefold()
    menu_names = {name.casefold(): name for name in MENU}
    requested_item = ALIASES.get(raw_item, menu_names.get(raw_item))

    if raw_item in {'croissant', 'croissants'}:
        return {
            'steps': steps,
            'valid': False,
            'decision': 'invalid',
            'reason': 'Croissants are no longer made and cannot be ordered.',
            'requested_item': raw_item,
        }
    if requested_item is None:
        return {
            'steps': steps,
            'valid': False,
            'decision': 'invalid',
            'reason': 'That product is not on the current menu.',
            'requested_item': raw_item,
        }

    quantity = state.get('quantity')
    try:
        parsed_quantity = int(quantity)
        if isinstance(quantity, bool) or str(quantity).strip() != str(parsed_quantity) or parsed_quantity <= 0:
            raise ValueError
    except (TypeError, ValueError):
        return {
            'steps': steps,
            'valid': False,
            'decision': 'invalid',
            'reason': 'The quantity must be a positive whole number.',
            'requested_item': requested_item,
        }

    day = str(state.get('day', '')).strip().casefold()
    if day not in WEEKDAYS:
        return {
            'steps': steps,
            'valid': False,
            'decision': 'invalid',
            'reason': 'Please provide a valid day of the week.',
            'requested_item': requested_item,
        }

    return {
        'steps': steps,
        'valid': True,
        'requested_item': requested_item,
        'quantity': parsed_quantity,
        'day': day.title(),
        'decision': 'ready_for_inventory_check',
    }


def check_inventory(state: OrderState) -> dict:
    steps = add_step(state, 'check_inventory')
    if not state.get('valid'):
        return {'steps': steps}

    requested_item = state['requested_item']
    if INVENTORY[requested_item] >= state['quantity']:
        return {
            'steps': steps,
            'selected_item': requested_item,
            'decision': 'ready',
            'substitution': False,
        }
    return {'steps': steps, 'decision': 'needs_substitution'}


def handle_substitution(state: OrderState) -> dict:
    steps = add_step(state, 'handle_substitution')
    if state.get('decision') != 'needs_substitution':
        return {'steps': steps}

    requested_item = state['requested_item']
    quantity = state['quantity']
    answers = state.get('replacement_answers', [])
    current_item = requested_item
    offers_made = 0

    while offers_made < 2:
        alternative = REPLACEMENT.get(current_item)
        if alternative is None:
            break
        current_item = alternative

        # Never offer an item unless stock covers the full requested quantity.
        if INVENTORY[alternative] < quantity:
            continue

        answer_index = offers_made
        offers_made += 1
        answer = answers[answer_index] if answer_index < len(answers) else None
        if isinstance(answer, bool):
            answer = 'yes' if answer else 'no'
        if answer is not None:
            answer = str(answer).strip().casefold()

        if answer in {'yes', 'y'}:
            return {
                'steps': steps,
                'selected_item': alternative,
                'decision': 'ready',
                'substitution': True,
                'offer_item': alternative,
                'offer_number': offers_made,
            }
        if answer in {'no', 'n'}:
            continue

        # No customer answer yet: return an offer without booking or charging.
        return {
            'steps': steps,
            'decision': 'awaiting_replacement_approval',
            'offer_item': alternative,
            'offer_number': offers_made,
            'substitution': False,
        }

    return {'steps': steps, 'decision': 'cancelled', 'substitution': False}


def choose_delivery(state: OrderState) -> dict:
    steps = add_step(state, 'choose_delivery')
    city = str(state.get('city', '')).strip()
    if city.casefold() == 'jerusalem':
        return {
            'steps': steps,
            'delivery': 'delivery',
            'delivery_fee': 20,
            'outside_delivery_area': False,
        }
    return {
        'steps': steps,
        'delivery': 'pickup',
        'delivery_fee': 0,
        'outside_delivery_area': bool(city),
    }


def calculate_price(state: OrderState) -> dict:
    steps = add_step(state, 'calculate_price')
    goods_subtotal = MENU[state['selected_item']] * state['quantity']
    discount = 0
    if goods_subtotal > 150 and str(state['day']).casefold() != 'friday':
        discount = round(goods_subtotal * 0.15, 2)
    total = round(goods_subtotal - discount + state.get('delivery_fee', 0), 2)
    return {
        'steps': steps,
        'goods_subtotal': goods_subtotal,
        'discount': discount,
        'total': total,
    }


def commit_inventory(state: OrderState) -> dict:
    steps = add_step(state, 'commit_inventory')
    sold_item = state['selected_item']
    INVENTORY[sold_item] -= state['quantity']
    return {'steps': steps, 'item': sold_item, 'status': 'accepted'}


def reject_order(state: OrderState) -> dict:
    # This route deliberately skips both delivery selection and price calculation.
    steps = add_step(state, 'reject_order')
    return {
        'steps': steps,
        'status': 'rejected',
        'item': None,
        'delivery': 'pickup',
        'delivery_fee': 0,
        'total': 0.0,
    }


def write_customer_reply(state: OrderState) -> dict:
    steps = add_step(state, 'write_customer_reply')
    requested_item = state.get('requested_item', state.get('item', 'that item'))
    city = str(state.get('city', '')).strip()

    if state.get('decision') == 'invalid':
        message = (f"Hi there! I couldn't understand this order: {state.get('reason', 'please check the details')}. "
                   "Please send it again and I'll be happy to help. —Dana")
    elif state.get('decision') == 'awaiting_replacement_approval':
        alternative = state['offer_item']
        quantity_text = quantity_label(alternative, state['quantity'])
        delivery_note = (' If you accept, it will be delivered in Jerusalem for ₪20.'
                         if city.casefold() == 'jerusalem' else
                         (' We do not deliver outside Jerusalem, so an accepted order would be free shop pickup.'
                          if city else ' An accepted order would be free shop pickup.'))
        message = (f"Hi there! We're out of {requested_item}. Would you like {quantity_text} instead "
                   f"at ₪{MENU[alternative]} each?{delivery_note} Please reply yes or no; your order is not placed until you accept. —Dana")
    elif state.get('decision') == 'cancelled':
        quantity_text = quantity_label(requested_item, state.get('quantity', 1)) if requested_item in MENU else 'your order'
        message = (f"I'm so sorry, but we couldn't fill {quantity_text} with an available replacement "
                   "after up to two offers. I've cancelled it. —Dana")
    else:
        selected_item = state['item']
        quantity_text = quantity_label(selected_item, state['quantity'])
        substitution_note = ''
        if state.get('substitution'):
            substitution_note = (f" We were out of {requested_item}; with your approval, I substituted "
                                 f"{quantity_text} and kept your original quantity.")
        city_note = ''
        if state.get('outside_delivery_area'):
            city_note = f' We do not deliver to {city}, so I switched your order to free shop pickup.'
        discount_note = (f" Your 15% baked-goods discount is ₪{state['discount']:.2f}."
                         if state.get('discount', 0) else '')
        delivery_note = ' Jerusalem delivery is ₪20.' if state.get('delivery') == 'delivery' else ''
        message = (f"Hi there! Thanks for ordering {quantity_text} from Dana's Bakery."
                   f"{substitution_note}{city_note}{discount_note}{delivery_note} "
                   f"Your total is ₪{state['total']:.2f}. See you soon! —Dana")

    return {'steps': steps, 'message': message}


## 4. Build the graph with conditional routes

Invalid orders route from validation straight to rejection. All valid orders pass through substitution checks. Orders with a shortage try up to two customer-approved replacements; if neither is accepted, they route straight to rejection. Only accepted orders reach delivery and pricing.


In [ ]:
builder = StateGraph(OrderState)
builder.add_node('validate_order', validate_order)
builder.add_node('check_inventory', check_inventory)
builder.add_node('handle_substitution', handle_substitution)
builder.add_node('choose_delivery', choose_delivery)
builder.add_node('calculate_price', calculate_price)
builder.add_node('commit_inventory', commit_inventory)
builder.add_node('reject_order', reject_order)
builder.add_node('write_customer_reply', write_customer_reply)

builder.add_edge(START, 'validate_order')
builder.add_conditional_edges(
    'validate_order',
    lambda state: 'continue' if state.get('valid') else 'reject',
    {'continue': 'check_inventory', 'reject': 'reject_order'},
)
# Every valid order passes through substitution checks, even when stock is sufficient.
builder.add_edge('check_inventory', 'handle_substitution')
builder.add_conditional_edges(
    'handle_substitution',
    lambda state: 'ready' if state.get('decision') == 'ready' else 'reject',
    {'ready': 'choose_delivery', 'reject': 'reject_order'},
)
builder.add_edge('choose_delivery', 'calculate_price')
builder.add_edge('calculate_price', 'commit_inventory')
builder.add_edge('commit_inventory', 'write_customer_reply')
builder.add_edge('reject_order', 'write_customer_reply')
builder.add_edge('write_customer_reply', END)

graph = builder.compile()


## 5. Render the graph layout

The graph is rendered as a Mermaid PNG inside this notebook.


In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## 6. Run the requested example

The exact input below finds babka out of stock. Because no replacement answer was provided, the graph rejects this invocation without booking it and asks whether the customer wants rugelach. All six required result fields are returned.


In [ ]:
sample = graph.invoke({"item": "babka", "quantity": 2, "day": "Thursday", "city": "Jerusalem"})
REQUIRED_KEYS = ['status', 'item', 'delivery', 'total', 'message', 'steps']
{key: sample[key] for key in REQUIRED_KEYS}


## 7. Approve the offered replacement

Pass the customer's answer in replacement_answers. The next invocation rechecks inventory; yes accepts the first eligible offer. To decline the first and accept the second, pass ['no', 'yes']. Two no answers cancel the order.


In [ ]:
approved_sample = graph.invoke({
    'item': 'babka',
    'quantity': 2,
    'day': 'Thursday',
    'city': 'Jerusalem',
    'replacement_answers': ['yes'],
})
{key: approved_sample[key] for key in REQUIRED_KEYS}


## 8. Run business-logic path tests

These independent test orders cover valid stock, first and second replacement acceptance, exhausted stock, invalid input, pickup outside Jerusalem, Friday discount exclusion, and the eligible non-Friday discount. Each test starts with a fresh copy of today's inventory, and the starting inventory is restored after the suite.


In [ ]:
TEST_CASES = [
    {
        'name': 'standard valid order with Jerusalem delivery',
        'order': {'item': 'challah', 'quantity': 2, 'day': 'Thursday', 'city': 'Jerusalem'},
        'status': 'accepted', 'item': 'challah', 'delivery': 'delivery', 'total': 56,
        'must_include': ['handle_substitution', 'choose_delivery', 'calculate_price', 'commit_inventory'],
    },
    {
        'name': 'first replacement accepted',
        'order': {'item': 'babka', 'quantity': 2, 'day': 'Thursday', 'city': '', 'replacement_answers': ['yes']},
        'status': 'accepted', 'item': 'box of rugelach', 'delivery': 'pickup', 'total': 60,
        'must_include': ['handle_substitution', 'choose_delivery', 'calculate_price'],
    },
    {
        'name': 'second replacement accepted after declining the first',
        'order': {'item': 'babka', 'quantity': 2, 'day': 'Thursday', 'city': 'Jerusalem', 'replacement_answers': ['no', 'yes']},
        'status': 'accepted', 'item': 'sourdough loaf', 'delivery': 'delivery', 'total': 70,
        'must_include': ['handle_substitution', 'choose_delivery', 'calculate_price'],
    },
    {
        'name': 'replacement offer waits for customer approval',
        'order': {'item': 'babka', 'quantity': 2, 'day': 'Thursday', 'city': 'Jerusalem'},
        'status': 'rejected', 'item': None, 'delivery': 'pickup', 'total': 0,
        'must_include': ['handle_substitution', 'reject_order'],
        'must_exclude': ['choose_delivery', 'calculate_price', 'commit_inventory'],
    },
    {
        'name': 'total stock cannot cover the requested quantity',
        'order': {'item': 'babka', 'quantity': 18, 'day': 'Monday', 'city': 'Jerusalem'},
        'status': 'rejected', 'item': None, 'delivery': 'pickup', 'total': 0,
        'must_include': ['handle_substitution', 'reject_order'],
        'must_exclude': ['choose_delivery', 'calculate_price', 'commit_inventory'],
    },
    {
        'name': 'invalid product rejected at validation',
        'order': {'item': 'pizza', 'quantity': 1, 'day': 'Monday', 'city': 'Jerusalem'},
        'status': 'rejected', 'item': None, 'delivery': 'pickup', 'total': 0,
        'must_include': ['validate_order', 'reject_order'],
        'must_exclude': ['check_inventory', 'handle_substitution', 'choose_delivery', 'calculate_price'],
    },
    {
        'name': 'outside Jerusalem converted to free pickup',
        'order': {'item': 'challah', 'quantity': 2, 'day': 'Tuesday', 'city': 'Tel Aviv'},
        'status': 'accepted', 'item': 'challah', 'delivery': 'pickup', 'total': 36,
        'must_include': ['choose_delivery', 'calculate_price'],
    },
    {
        'name': 'Friday large order has no discount',
        'order': {'item': 'challah', 'quantity': 10, 'day': 'Friday', 'city': 'Jerusalem'},
        'status': 'accepted', 'item': 'challah', 'delivery': 'delivery', 'total': 200,
        'discount': 0,
    },
    {
        'name': 'large non-Friday order receives 15 percent discount',
        'order': {'item': 'challah', 'quantity': 10, 'day': 'Thursday', 'city': ''},
        'status': 'accepted', 'item': 'challah', 'delivery': 'pickup', 'total': 153,
        'discount': 27,
    },
    {
        'name': 'two replacement offers declined and order rejected',
        'order': {'item': 'babka', 'quantity': 2, 'day': 'Monday', 'city': '', 'replacement_answers': ['no', 'no']},
        'status': 'rejected', 'item': None, 'delivery': 'pickup', 'total': 0,
        'must_include': ['handle_substitution', 'reject_order'],
        'must_exclude': ['choose_delivery', 'calculate_price'],
    },
    {
        'name': 'discontinued croissant rejected at validation',
        'order': {'item': 'croissant', 'quantity': 1, 'day': 'Monday', 'city': ''},
        'status': 'rejected', 'item': None, 'delivery': 'pickup', 'total': 0,
        'must_exclude': ['check_inventory', 'choose_delivery', 'calculate_price'],
    },
    {
        'name': 'delivery fee is not included in the discount calculation',
        'order': {'item': 'challah', 'quantity': 9, 'day': 'Thursday', 'city': 'Jerusalem'},
        'status': 'accepted', 'item': 'challah', 'delivery': 'delivery', 'total': 157.7,
        'discount': 24.3,
    },
]

EXPECTED_PATHS = {
    'standard valid order with Jerusalem delivery': ['validate_order', 'check_inventory', 'handle_substitution', 'choose_delivery', 'calculate_price', 'commit_inventory', 'write_customer_reply'],
    'first replacement accepted': ['validate_order', 'check_inventory', 'handle_substitution', 'choose_delivery', 'calculate_price', 'commit_inventory', 'write_customer_reply'],
    'second replacement accepted after declining the first': ['validate_order', 'check_inventory', 'handle_substitution', 'choose_delivery', 'calculate_price', 'commit_inventory', 'write_customer_reply'],
    'replacement offer waits for customer approval': ['validate_order', 'check_inventory', 'handle_substitution', 'reject_order', 'write_customer_reply'],
    'total stock cannot cover the requested quantity': ['validate_order', 'check_inventory', 'handle_substitution', 'reject_order', 'write_customer_reply'],
    'invalid product rejected at validation': ['validate_order', 'reject_order', 'write_customer_reply'],
    'outside Jerusalem converted to free pickup': ['validate_order', 'check_inventory', 'handle_substitution', 'choose_delivery', 'calculate_price', 'commit_inventory', 'write_customer_reply'],
    'Friday large order has no discount': ['validate_order', 'check_inventory', 'handle_substitution', 'choose_delivery', 'calculate_price', 'commit_inventory', 'write_customer_reply'],
    'large non-Friday order receives 15 percent discount': ['validate_order', 'check_inventory', 'handle_substitution', 'choose_delivery', 'calculate_price', 'commit_inventory', 'write_customer_reply'],
    'two replacement offers declined and order rejected': ['validate_order', 'check_inventory', 'handle_substitution', 'reject_order', 'write_customer_reply'],
    'discontinued croissant rejected at validation': ['validate_order', 'reject_order', 'write_customer_reply'],
    'delivery fee is not included in the discount calculation': ['validate_order', 'check_inventory', 'handle_substitution', 'choose_delivery', 'calculate_price', 'commit_inventory', 'write_customer_reply'],
}

starting_inventory_snapshot = INVENTORY.copy()
try:
    for case in TEST_CASES:
        INVENTORY.clear()
        INVENTORY.update(STARTING_INVENTORY)
        result = graph.invoke(case['order'])

        assert all(key in result for key in REQUIRED_KEYS), f"{case['name']}: missing required output key"
        assert result['steps'] == EXPECTED_PATHS[case['name']], (
            f"{case['name']}: expected path {EXPECTED_PATHS[case['name']]}, got {result['steps']}"
        )
        for key in ('status', 'item', 'delivery', 'total'):
            assert result[key] == case[key], (
                f"{case['name']}: expected {key}={case[key]!r}, got {result[key]!r}"
            )
        for step in case.get('must_include', []):
            assert step in result['steps'], f"{case['name']}: expected step {step!r}"
        for step in case.get('must_exclude', []):
            assert step not in result['steps'], f"{case['name']}: unexpected step {step!r}"
        if 'discount' in case:
            assert result.get('discount') == case['discount'], (
                f"{case['name']}: expected discount {case['discount']}, got {result.get('discount')}"
            )
        print(f"PASS: {case['name']} | status={result['status']} | total=₪{result['total']}")
finally:
    INVENTORY.clear()
    INVENTORY.update(starting_inventory_snapshot)

print(f"All {len(TEST_CASES)} order-path tests passed.")
